# PageRank on the AS-level Internet graph

**Irene Sarrias Perez**

Internet and Networked Economy, MATT, ETSETB, UPC. Programming assignment, Project 3.4: PageRank on a non-web graph.

## 0. Summary

- The CAIDA AS Relationships snapshot of 1 September 2026 gives 80,510 autonomous systems (ASes) and 884,790 directed edges. An edge points from a customer to its provider, and each peering link becomes one edge in each direction.
- PageRank is implemented from scratch with the power method on a sparse matrix. Dangling nodes (ASes with no out-link) redistribute their mass uniformly. The code is checked against exact fractions, a hand-derived fixture, and the lecture examples.
- At damping 0.85 the power method converges in 69 iterations to a residual below 1e-10. Of the top 20 ASes, 16 have more peers than customers and 4 belong to the clique. In all, 14 of the 18 clique members are in the top 100.
- PageRank and in-degree: the two rankings agree on 9 of the top 10, 44 of the top 50 and 86 of the top 100 ASes. They disagree where an AS has many in-links from networks with thousands of out-links, each of which passes on little rank.
- With peering links removed, 494 ASes have no out-link, all 18 clique members among them, and they hold 22.5% of the PageRank mass. The same solver handles it.

The notebook needs numpy, scipy, pandas and matplotlib, and nothing else outside the Python standard library.

In [ ]:
from __future__ import annotations

import bz2
import hashlib
import os
import shutil
import time
import urllib.request
from dataclasses import dataclass
from fractions import Fraction
from pathlib import Path
from typing import Any, Final, Iterable, Literal, Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scipy.stats
from scipy.sparse.csgraph import connected_components

pd.set_option("display.float_format", "{:.6g}".format)
pd.set_option("display.width", 120)
T0 = time.perf_counter()

## 1. Problem and data

An autonomous system (AS) is a network under one administrative control: an Internet service provider, a university, a content company. ASes exchange traffic under commercial agreements of two kinds. In a **customer-provider** link the customer pays the provider for transit. In a **peer-to-peer** link two ASes exchange traffic between themselves and their customers, usually without payment.

The data is the CAIDA *AS Relationships* dataset (serial-1), snapshot 2026-09-01. It lists one relationship per line, inferred from BGP routing paths seen at many vantage points (Luckie et al., *AS Relationships, Customer Cones, and Validation*, IMC 2013). This is a non-web graph: the nodes are network operators, and the links are commercial agreements instead of hyperlinks between pages.

The cell below downloads the two files of the snapshot into `caida_data/`, next to this notebook, and checks their md5 sums. A file that is already there with the right sum is not downloaded again. A wrong sum stops the notebook with an error.

In [ ]:
DATA_DIR = Path("caida_data")
BASE_URL = "https://publicdata.caida.org/datasets/as-relationships/serial-1/"
SNAPSHOT_MD5 = {
    "20260901.as-rel.txt.bz2": "f4cba16613623eb1bc67355d1dc25213",
    "20260901.ppdc-ases.txt.bz2": "1c6b0c4c62a16d9bfa6c8fb75e6cd03d",
}


def md5_of(path: Path, chunk: int = 1 << 20) -> str:
    """md5 of a file, read in chunks so the whole file is never in memory."""
    digest = hashlib.md5()
    with open(path, "rb") as handle:
        while block := handle.read(chunk):
            digest.update(block)
    return digest.hexdigest()


def fetch_caida(data_dir: Path = DATA_DIR) -> dict[str, Path]:
    """Download the snapshot files that are missing and verify every md5.

    The download goes to a temporary name and is renamed only after the md5
    matches, so an interrupted or corrupted download never passes for a
    good file on the next run.
    """
    data_dir.mkdir(parents=True, exist_ok=True)
    paths: dict[str, Path] = {}
    for name, expected in SNAPSHOT_MD5.items():
        path = data_dir / name
        if path.exists() and md5_of(path) == expected:
            print(f"{name}: already present, md5 OK")
        else:
            print(f"{name}: downloading")
            partial = path.with_name(name + ".part")
            with urllib.request.urlopen(BASE_URL + name, timeout=120) as response:
                with open(partial, "wb") as out:
                    shutil.copyfileobj(response, out)
            found = md5_of(partial)
            if found != expected:
                partial.unlink()
                raise RuntimeError(
                    f"{name}: md5 is {found}, expected {expected}. "
                    "The file at CAIDA has changed or the download is corrupt."
                )
            partial.replace(path)
            print(f"{name}: downloaded, md5 OK")
        paths[name] = path
    return paths


paths = fetch_caida()
AS_REL = paths["20260901.as-rel.txt.bz2"]
PPDC = paths["20260901.ppdc-ases.txt.bz2"]
for name, path in paths.items():
    print(f"{name}: {path.stat().st_size:,} bytes, md5 {md5_of(path)}")

The header of the file states what the data is and what was inferred. The first lines:

In [ ]:
header, data = [], []
with bz2.open(AS_REL, "rt", encoding="ascii") as handle:
    for line in handle:
        (header if line.startswith("#") else data).append(line.rstrip())
        if len(data) == 3:
            break
for line in header[:4]:
    print(line[:100])
print(f"... {len(header)} comment lines before the data, then one relationship per line:")
for line in data:
    print(line)

A relationship line is `provider|customer|-1` for a customer-provider link and `peer|peer|0` for a peering link.

### Modelling the data as a directed graph

Some choices are facts from the data and some are decisions.

**Facts.** The nodes are the ASNs that appear in a relationship. A customer-provider line says who is the customer and who is the provider. A peer-to-peer line is symmetric. The header names an input clique of 18 transit-free ASes that the inference treats as the top of the hierarchy.

**Decisions.**

1. *Vote direction.* PageRank needs an edge to mean "endorses". We let a customer endorse its provider: the edge runs customer to provider, so rank flows up the hierarchy towards the networks that many others pay for transit.
2. *Peering links.* A peering link says nothing about who endorses whom. We either add an edge in each direction (`both`) or leave peering out (`none`). The counts in section 5 show why this matters: most edges are peering edges, and without them 494 ASes have no out-link, the whole clique included. The decision is argued in section 5.
3. *Duplicates and self-loops* are removed and counted. Every AS is kept, including the 28 IXP ASes (internet exchange points) that the header lists.

## 2. Method

**From the graph to the Google matrix.** Number the $N$ ASes $1,\dots,N$. Let $A$ be the adjacency matrix, with $A_{ij}=1$ when $i$ links to $j$. The course builds PageRank in four steps, $A \to H \to S \to G$:

- $H$ divides each row of $A$ by its sum, so row $i$ spreads the vote of $i$ equally over its out-links. A dangling AS has a zero row.
- $S = H + \frac{1}{N}\, w\,\mathbf 1^T$, where $w_i = 1$ for dangling rows and 0 otherwise. A dangling AS now jumps to any AS with probability $1/N$, so every row of $S$ sums to 1.
- $G = \alpha S + (1-\alpha)\frac{1}{N}\mathbf 1\mathbf 1^T$. With probability $1-\alpha$ the surfer jumps to a uniformly random AS (teleportation).

PageRank is the row vector $\pi$ with $\pi^T = \pi^T G$ and $\pi^T\mathbf 1 = 1$. For $0<\alpha<1$ every entry of $G$ is positive, so $G$ is primitive. By the Perron-Frobenius theorem, $\pi$ exists, is unique and is strictly positive. The graph itself is not irreducible (section 5), so teleportation is what makes the answer unique.

**The update that is actually computed.** $G$ would have $80{,}510^2 \approx 6.5\times10^9$ entries, against 884,790 non-zeros in $H$. So $G$ is never built. One step of the power method is

$$\pi^{(k+1)T} = \alpha\,\pi^{(k)T}H + \big(\alpha\,\pi^{(k)T}w + 1-\alpha\big)\frac{\mathbf 1^T}{N},$$

which costs one sparse product. In the code the matrix is stored transposed, because the step is $x \leftarrow \alpha P^T x + \dots$ and a CSR matrix-vector product is fastest on the stored matrix.

**Dangling treatment.** The term $\alpha\,\pi^T w$ is the mass sitting on dangling ASes, and it is spread uniformly. With uniform teleportation this gives the same vector as leaving $H$ substochastic, solving the linear system, and normalising the result to sum 1. Section 4 checks this on a fixture.

**Stopping rule.** Iterate until $\|\pi^{(k+1)}-\pi^{(k)}\|_1 < 10^{-10}$. Each step shrinks the error by at least a factor $\alpha$, so about $\lceil \ln(10^{-10}) / \ln \alpha \rceil$ steps are enough, 142 for $\alpha=0.85$. Section 9.2 shows that the actual counts are lower.

## 3. Implementation

The cells in this section define everything the later sections use. Each is a single block of code, shown here in full. Nothing outside the Python standard library, NumPy, SciPy, pandas and matplotlib is used, and no PageRank routine comes from a library.

**Graph.** The graph is stored as the transposed, row-normalised matrix $P^T$ in CSR form, because the power step multiplies by $P^T$. Out-degree and in-degree are kept as arrays, and a dangling node is one with out-degree 0. Duplicate edges are dropped, not summed into weights, and self-loops are removed. Both counts are kept.

In [ ]:
@dataclass(frozen=True)
class DirectedGraph:
    """A directed graph as the sparse structure PageRank needs.

    Attributes:
        n: Number of nodes. Node ids are ``0 .. n-1``.
        transpose: ``P^T`` in CSR form, shape ``(n, n)``. The entry
            ``transpose[j, i]`` equals ``1 / out_degree[i]`` for every edge
            ``i -> j``. Columns of dangling nodes are entirely zero. Stored
            transposed on purpose: the power step multiplies ``P^T x`` and CSR
            mat-vec on the stored matrix is the fast path.
        out_degree: Out-degree of each node after cleaning, int64, shape ``(n,)``.
        in_degree: In-degree of each node after cleaning, int64, shape ``(n,)``.
        n_duplicates_dropped: Number of exact duplicate ``(tail, head)`` pairs
            removed. Duplicates are dropped, never summed into a weight.
        n_self_loops_dropped: Number of self-loops removed.
    """

    n: int
    transpose: sp.csr_matrix
    out_degree: np.ndarray
    in_degree: np.ndarray
    n_duplicates_dropped: int
    n_self_loops_dropped: int

    @property
    def dangling(self) -> np.ndarray:
        """Boolean mask of nodes with no outgoing edge, shape ``(n,)``."""
        return self.out_degree == 0

    @property
    def n_edges(self) -> int:
        """Number of edges after removing self-loops and duplicates."""
        return int(self.out_degree.sum())


def from_edge_list(
    edges: Iterable[tuple[int, int]] | np.ndarray,
    n: int,
    *,
    drop_self_loops: bool = True,
) -> DirectedGraph:
    """Build a :class:`DirectedGraph` from an edge list and a node count.

    Args:
        edges: Either any iterable of ``(tail, head)`` integer pairs
            (``tail -> head``), or an ``(m, 2)`` integer array of such pairs.
            The array form is a fast path that skips the per-pair Python loop,
            so the real graph's edge array loads in well under a second.
        n: Number of nodes. Every endpoint must satisfy ``0 <= id < n``.
        drop_self_loops: If true (default), remove edges with ``tail == head``
            and count them in ``n_self_loops_dropped``.

    Returns:
        A :class:`DirectedGraph` whose ``transpose`` is the row-normalised
        transition matrix transposed. Self-loops (optionally) and exact
        duplicate pairs are removed and counted; duplicate edges are never
        silently summed into a larger weight.

    Raises:
        ValueError: If ``n < 0``, if an array ``edges`` is not shaped
            ``(m, 2)``, or if any endpoint is out of range (the offending pair
            is named in the message).
    """
    if n < 0:
        raise ValueError(f"n must be non-negative, got {n}")

    if isinstance(edges, np.ndarray):
        # Fast path for the real graph: an (m, 2) array (the builder's
        # `edges`) is taken as is, skipping the per-pair Python loop that
        # `np.fromiter` would run. An 884k-edge array loads in well under a
        # second this way.
        if edges.ndim != 2 or edges.shape[1] != 2:
            raise ValueError(
                "an edge array must have shape (m, 2), got "
                f"{edges.shape}"
            )
        arr = np.asarray(edges, dtype=np.int64)
    else:
        arr = np.fromiter(
            (coord for pair in edges for coord in pair),
            dtype=np.int64,
        )
        if arr.size % 2 != 0:
            raise ValueError("edges must be an iterable of (tail, head) pairs")
        arr = arr.reshape(-1, 2)

    # Validate ranges before doing any arithmetic, naming the offending pair.
    if arr.size:
        bad = (arr < 0) | (arr >= n)
        if bad.any():
            row = int(np.argmax(bad.any(axis=1)))
            tail, head = int(arr[row, 0]), int(arr[row, 1])
            raise ValueError(
                f"edge ({tail}, {head}) has an endpoint outside [0, {n})"
            )

    # Remove self-loops first, then exact duplicate pairs; count both.
    n_self_loops_dropped = 0
    if drop_self_loops and arr.size:
        keep = arr[:, 0] != arr[:, 1]
        n_self_loops_dropped = int((~keep).sum())
        arr = arr[keep]

    n_duplicates_dropped = 0
    if arr.size:
        unique = np.unique(arr, axis=0)
        n_duplicates_dropped = int(arr.shape[0] - unique.shape[0])
        arr = unique

    tails = arr[:, 0]
    heads = arr[:, 1]

    out_degree = np.bincount(tails, minlength=n).astype(np.int64)
    in_degree = np.bincount(heads, minlength=n).astype(np.int64)

    # transpose[head, tail] = 1 / out_degree[tail]. Every surviving edge has a
    # non-dangling tail by construction, so out_degree[tails] is never zero.
    data = 1.0 / out_degree[tails].astype(np.float64)
    transpose = sp.coo_matrix(
        (data, (heads, tails)),
        shape=(n, n),
        dtype=np.float64,
    ).tocsr()

    return DirectedGraph(
        n=n,
        transpose=transpose,
        out_degree=out_degree,
        in_degree=in_degree,
        n_duplicates_dropped=n_duplicates_dropped,
        n_self_loops_dropped=n_self_loops_dropped,
    )

**Power iteration.** The state is a float64 vector, because the residual is compared with $10^{-10}$ and the vector holds values near $10^{-6}$. After each step the code checks that the vector still sums to 1 and raises if it does not. A wrong dangling term loses or creates mass, and this check turns that into an error instead of a quietly wrong ranking.

In [ ]:
@dataclass(frozen=True)
class PageRankResult:
    """Outcome of a power-iteration run.

    Attributes:
        vector: PageRank vector, float64, shape ``(n,)``, sums to 1.
        iterations: Number of update steps performed.
        residuals: L1 norm ``||x_{k+1} - x_k||_1`` for each step performed, in
            order. One entry per iteration.
        converged: True if the run stopped because the residual fell below
            ``tol``; False if it hit ``max_iter`` first.
        alpha: The damping factor used.
    """

    vector: np.ndarray
    iterations: int
    residuals: tuple[float, ...]
    converged: bool
    alpha: float


def _as_distribution(
    candidate: np.ndarray | None,
    n: int,
    name: str,
) -> np.ndarray:
    """Return a validated probability vector of length ``n``.

    ``None`` yields the uniform distribution. Otherwise the vector must be
    non-negative and sum to 1 within 1e-12; it is normalised if the sum is
    within tolerance of 1, and rejected otherwise.
    """
    if candidate is None:
        return np.full(n, 1.0 / n, dtype=np.float64)

    vec = np.asarray(candidate, dtype=np.float64)
    if vec.shape != (n,):
        raise ValueError(f"{name} must have shape ({n},), got {vec.shape}")
    if np.any(vec < 0):
        raise ValueError(f"{name} must be non-negative")

    total = float(vec.sum())
    if abs(total - 1.0) > 1e-12:
        raise ValueError(f"{name} must sum to 1 (sums to {total!r})")
    return vec / total


def pagerank_power(
    graph: DirectedGraph,
    alpha: float = 0.85,
    *,
    tol: float = 1e-10,
    max_iter: int = 1000,
    teleport: np.ndarray | None = None,
    x0: np.ndarray | None = None,
) -> PageRankResult:
    """Compute PageRank of ``graph`` by the damped power method.

    Args:
        graph: The directed graph; its ``transpose`` is ``P^T``.
        alpha: Damping factor, ``0 <= alpha < 1``.
        tol: Convergence threshold on the L1 residual between iterates.
        max_iter: Maximum number of iterations before giving up (returns
            ``converged=False``; does not raise).
        teleport: Teleport distribution ``v``; uniform ``1/n`` when ``None``.
        x0: Initial iterate; the teleport distribution when ``None``.

    Returns:
        A :class:`PageRankResult`. ``vector`` sums to 1.

    Raises:
        ValueError: If ``graph.n == 0``, ``alpha`` is outside ``[0, 1)``, or
            ``teleport``/``x0`` is not a valid probability vector.
    """
    n = graph.n
    if n == 0:
        raise ValueError("cannot run PageRank on an empty graph (n == 0)")
    if not (0.0 <= alpha < 1.0):
        raise ValueError(f"alpha must satisfy 0 <= alpha < 1, got {alpha!r}")

    v = _as_distribution(teleport, n, "teleport")
    x = _as_distribution(x0, n, "x0").copy()

    dangling = graph.dangling
    pt = graph.transpose

    residuals: list[float] = []
    converged = False
    for _ in range(max_iter):
        m = float(x[dangling].sum())
        x_new = alpha * (pt @ x) + (alpha * m + (1.0 - alpha)) * v
        r = float(np.abs(x_new - x).sum())
        residuals.append(r)
        x = x_new
        if r < tol:
            converged = True
            break

    # Analytically the iterate sums to 1 at every step (that is the point of
    # the dangling repair), so this only removes floating-point drift. If the
    # sum has drifted meaningfully the update is wrong, so check first. An
    # explicit raise rather than an assert: asserts are stripped under -O,
    # and this is an invariant of the library, not a debugging aid.
    total = float(x.sum())
    if abs(total - 1.0) >= 1e-9:
        raise RuntimeError(
            f"iterate mass drifted to {total!r}; the dangling update is wrong"
        )
    x = x / total

    return PageRankResult(
        vector=x,
        iterations=len(residuals),
        residuals=tuple(residuals),
        converged=converged,
        alpha=alpha,
    )

**Reading the data.** The parser keeps what the file says (relationship rows, the clique, the IXP list) and makes no modelling choice. `build_edge_list` is where the two decisions of section 1 live, as the parameters `peering` and `vote`. ASNs are mapped to node indices with a binary search on the sorted ASN array.

In [ ]:
#: Relationship code for a customer-provider link: the first field is the
#: provider, the second its customer. From CAIDA's description of the format.
C2P: Final = -1


#: Relationship code for a peer-to-peer link: the two fields are peers.
P2P: Final = 0


@dataclass(frozen=True)
class AsRelSnapshot:
    """One ``as-rel`` snapshot, parsed but not yet modelled.

    Holds exactly what the file states, with no edge-direction or peering
    choice applied. The three parallel arrays ``relationships`` and ``step``
    describe the same relationship lines in file order.

    Attributes:
        path: The path the snapshot was read from.
        clique: ASNs on the ``# input clique:`` line, in file order. These are
            the transit-free core the inference forces (CAIDA's description of
            the format).
        ixp_asns: ASNs on the ``# IXP ASes:`` line, in file order.
        sources: The text after ``# source:`` on each source line, verbatim and
            in file order.
        relationships: ``(m, 3)`` int64 array; columns are ``as1``, ``as2`` and
            ``rel`` (``C2P`` or ``P2P``). For a ``C2P`` row ``as1`` is the
            provider and ``as2`` the customer; for a ``P2P`` row they are peers.
        step: ``(m,)`` int64 array; the ``N`` of the last ``# step N`` comment
            seen before each relationship, or 0 for any relationship appearing
            before the first step comment.
        n_comment_lines: Total count of lines beginning with ``#``, including
            the ones not matched by any known prefix.
    """

    path: str
    clique: tuple[int, ...]
    ixp_asns: tuple[int, ...]
    sources: tuple[str, ...]
    relationships: np.ndarray
    step: np.ndarray
    n_comment_lines: int

    @property
    def n_c2p(self) -> int:
        """Number of customer-provider relationships."""
        return int(np.count_nonzero(self.relationships[:, 2] == C2P))

    @property
    def n_p2p(self) -> int:
        """Number of peer-to-peer relationships."""
        return int(np.count_nonzero(self.relationships[:, 2] == P2P))


def _open_text(path: str | os.PathLike):
    """Open ``path`` as a text stream, transparently handling bz2.

    The snapshots ship bz2-compressed, but tests and small fixtures use plain
    text, so the suffix picks the opener. The file is ASCII.
    """
    name = os.fspath(path)
    if name.endswith(".bz2"):
        return bz2.open(name, "rt", encoding="ascii")
    return open(name, "rt", encoding="ascii")


def read_as_rel(path: str | os.PathLike) -> AsRelSnapshot:
    """Parse an ``as-rel`` file into an :class:`AsRelSnapshot`.

    Reads a ``serial-1`` relationship file (bz2 when the suffix is ``.bz2``,
    plain text otherwise). Comment lines are classified by their known
    prefixes and counted; every other ``#`` line is counted but ignored. Blank
    lines are ignored and not counted. Relationship lines are collected into
    integer arrays in file order, each tagged with the step it appeared under.

    Args:
        path: Path to the ``as-rel`` file.

    Returns:
        The parsed snapshot. ``relationships`` and ``step`` are parallel arrays
        in file order.

    Raises:
        ValueError: On a relationship value other than ``C2P`` or ``P2P``, a
            self-relationship (``as1 == as2``), or a malformed line (not three
            ``|``-separated integers). The message names the 1-based line
            number.
    """
    clique: tuple[int, ...] = ()
    ixp_asns: tuple[int, ...] = ()
    sources: list[str] = []
    rows: list[tuple[int, int, int]] = []
    steps: list[int] = []

    n_comment_lines = 0
    current_step = 0

    with _open_text(path) as handle:
        for k, raw in enumerate(handle, start=1):
            line = raw.rstrip("\n")

            if line.startswith("#"):
                n_comment_lines += 1
                if line.startswith("# input clique:"):
                    clique = tuple(int(t) for t in line[len("# input clique:"):].split())
                elif line.startswith("# IXP ASes:"):
                    ixp_asns = tuple(int(t) for t in line[len("# IXP ASes:"):].split())
                elif line.startswith("# source:"):
                    sources.append(line[len("# source:"):])
                elif line.startswith("# step "):
                    # "# step N: description" -- N is the token before the colon.
                    tag = line[len("# step "):].split(":", 1)[0].strip()
                    current_step = int(tag)
                # Any other comment line is counted above and otherwise ignored.
                continue

            if not line.strip():
                continue

            fields = line.split("|")
            if len(fields) != 3:
                raise ValueError(
                    f"line {k}: malformed relationship, expected three "
                    f"'|'-separated fields, got {len(fields)}"
                )
            try:
                as1, as2, rel = (int(f) for f in fields)
            except ValueError:
                raise ValueError(
                    f"line {k}: malformed relationship, non-integer field in {line!r}"
                ) from None

            if rel not in (C2P, P2P):
                raise ValueError(f"line {k}: unexpected relationship {rel!r}")
            if as1 == as2:
                raise ValueError(f"line {k}: self-relationship for AS {as1}")

            rows.append((as1, as2, rel))
            steps.append(current_step)

    relationships = np.array(rows, dtype=np.int64).reshape(-1, 3)
    step = np.array(steps, dtype=np.int64)

    return AsRelSnapshot(
        path=os.fspath(path),
        clique=clique,
        ixp_asns=ixp_asns,
        sources=tuple(sources),
        relationships=relationships,
        step=step,
        n_comment_lines=n_comment_lines,
    )


@dataclass(frozen=True)
class EdgeListBuild:
    """A snapshot turned into node indices and a directed edge list.

    The three parameters that distinguish one build from another are recorded
    alongside the result so a caller can report what each one did, and the
    counts make the effect of exclusion and of dropping peering auditable.

    Attributes:
        asns: ``(n,)`` int64, the ASNs surviving exclusion, sorted ascending.
            Node index ``i`` is ASN ``asns[i]``; the sorted order is what makes
            :func:`index_of` a binary search.
        edges: ``(k, 2)`` int64 of node indices, ``tail -> head`` in the vote
            direction. Duplicates and self-loops are *not* removed here; that is
            :func:`pagerank.graph.from_edge_list`'s job, which also counts them.
        peering: ``"both"`` or ``"none"``, the peering convention used.
        vote: ``"customer_to_provider"`` or ``"provider_to_customer"``, which
            way a customer-provider link points.
        excluded_asns: The ASNs excluded, sorted ascending.
        n_c2p_used: Customer-provider relationships that contributed an edge
            (i.e. survived exclusion).
        n_p2p_used: Peer-to-peer relationships that survived exclusion. Each
            contributes two edges under ``both`` and none under ``none``.
        n_relationships_excluded: Relationships dropped because an endpoint was
            in ``excluded_asns``.
        n_asns_dropped: ASNs present in the snapshot but absent from ``asns``,
            i.e. those that only ever appeared in excluded relationships.
        snapshot_path: The path of the snapshot this build came from.
    """

    asns: np.ndarray
    edges: np.ndarray
    peering: str
    vote: str
    excluded_asns: tuple[int, ...]
    n_c2p_used: int
    n_p2p_used: int
    n_relationships_excluded: int
    n_asns_dropped: int
    snapshot_path: str

    @property
    def n(self) -> int:
        """Number of nodes (distinct surviving ASNs)."""
        return int(self.asns.shape[0])


def index_of(asns: np.ndarray, query: np.ndarray) -> np.ndarray:
    """Map ASNs to node indices against a sorted ASN array.

    Args:
        asns: Sorted-ascending int64 array of ASNs (an :class:`EdgeListBuild`'s
            ``asns``).
        query: ASNs to look up.

    Returns:
        An int64 array ``pos`` with ``asns[pos] == query`` elementwise.

    Raises:
        KeyError: If any ASN in ``query`` is not present, naming the first one.
    """
    query = np.asarray(query, dtype=np.int64)
    pos = np.searchsorted(asns, query)
    # searchsorted can return n for values past the end; clip before indexing.
    clipped = np.clip(pos, 0, asns.shape[0] - 1)
    bad = asns[clipped] != query
    if np.any(bad):
        first = int(query[np.argmax(bad)])
        raise KeyError(f"ASN {first} is not in the index")
    return pos


def build_edge_list(
    snapshot: AsRelSnapshot,
    *,
    peering: Literal["both", "none"] = "both",
    vote: Literal[
        "customer_to_provider", "provider_to_customer"
    ] = "customer_to_provider",
    exclude_asns: Iterable[int] = (),
) -> EdgeListBuild:
    """Turn a snapshot into an index map and a directed edge list.

    Three modelling choices are parameters rather than defaults baked into the
    data, so every variant can be built and compared from one read:

    * ``vote`` -- which endpoint of a customer-provider link the edge points
      to. The default ``customer_to_provider`` is the reading argued for:
      a customer paying for transit endorses its provider, so the
      edge runs customer -> provider and mass flows up towards the core.
    * ``peering`` -- whether a peer-to-peer link contributes edges at all.
      ``both`` adds an edge each way (peering is symmetric); ``none`` adds
      nothing, which makes the transit-free clique dangle.
    * ``exclude_asns`` -- ASNs to drop entirely, with every relationship that
      touches them. Used to study the effect of the IXP ASes.

    Exclusion happens *before* the ASN set is built, so an AS that only ever
    appeared next to an excluded one disappears from the graph and is counted
    in ``n_asns_dropped``.

    Args:
        snapshot: The parsed snapshot.
        peering: Peering convention, ``"both"`` or ``"none"``.
        vote: Customer-provider edge direction.
        exclude_asns: ASNs to drop, with all their relationships.

    Returns:
        An :class:`EdgeListBuild`. Edges are node indices into ``asns``, not
        ASNs. Duplicates and self-loops are left in for ``from_edge_list`` to
        remove and count.

    Raises:
        ValueError: If ``peering`` or ``vote`` is not one of the allowed values.
    """
    if peering not in ("both", "none"):
        raise ValueError(f"peering must be 'both' or 'none', got {peering!r}")
    if vote not in ("customer_to_provider", "provider_to_customer"):
        raise ValueError(
            "vote must be 'customer_to_provider' or 'provider_to_customer', "
            f"got {vote!r}"
        )

    rel = snapshot.relationships
    as1, as2, kind = rel[:, 0], rel[:, 1], rel[:, 2]

    excluded = np.array(sorted(set(int(a) for a in exclude_asns)), dtype=np.int64)
    excluded_tuple = tuple(int(a) for a in excluded)

    # Distinct ASNs in the whole snapshot, for the dropped-count below.
    all_asns = np.unique(rel[:, :2])

    # Drop relationships touching an excluded ASN, before anything else.
    if excluded.size:
        touched = np.isin(as1, excluded) | np.isin(as2, excluded)
    else:
        touched = np.zeros(rel.shape[0], dtype=bool)
    n_relationships_excluded = int(touched.sum())
    keep = ~touched

    as1, as2, kind = as1[keep], as2[keep], kind[keep]

    # The surviving ASN set defines the node indices. Sorted so index_of works.
    if as1.size:
        asns = np.unique(np.concatenate([as1, as2]))
    else:
        asns = np.array([], dtype=np.int64)
    n_asns_dropped = int(all_asns.shape[0] - asns.shape[0])

    is_c2p = kind == C2P
    is_p2p = kind == P2P
    n_c2p_used = int(is_c2p.sum())
    n_p2p_used = int(is_p2p.sum())

    # Build edges as ASN pairs first, then map once to node indices.
    tails: list[np.ndarray] = []
    heads: list[np.ndarray] = []

    # Customer-provider rows are stored (provider, customer) = (as1, as2).
    prov, cust = as1[is_c2p], as2[is_c2p]
    if vote == "customer_to_provider":
        tails.append(cust)
        heads.append(prov)
    else:  # provider_to_customer
        tails.append(prov)
        heads.append(cust)

    # Peer-to-peer rows: symmetric under "both", absent under "none".
    if peering == "both":
        pa, pb = as1[is_p2p], as2[is_p2p]
        tails.append(pa)
        heads.append(pb)
        tails.append(pb)
        heads.append(pa)

    tail_asns = (
        np.concatenate(tails) if tails else np.array([], dtype=np.int64)
    )
    head_asns = (
        np.concatenate(heads) if heads else np.array([], dtype=np.int64)
    )

    if tail_asns.size:
        edges = np.column_stack(
            [index_of(asns, tail_asns), index_of(asns, head_asns)]
        ).astype(np.int64)
    else:
        edges = np.empty((0, 2), dtype=np.int64)

    return EdgeListBuild(
        asns=asns.astype(np.int64),
        edges=edges,
        peering=peering,
        vote=vote,
        excluded_asns=excluded_tuple,
        n_c2p_used=n_c2p_used,
        n_p2p_used=n_p2p_used,
        n_relationships_excluded=n_relationships_excluded,
        n_asns_dropped=n_asns_dropped,
        snapshot_path=snapshot.path,
    )

**Comparing rankings.** Ties are broken by node index, so a ranking is always a strict order and repeatable. Kendall's tau-b is used because in-degree is a small integer with many ties, and tau-b corrects for them.

In [ ]:
def ranking(scores: np.ndarray) -> np.ndarray:
    """Return node indices ordered by score, highest first.

    Ties are broken by ascending index, deterministically.

    Args:
        scores: Score per node, shape ``(n,)``.

    Returns:
        An int64 array of node indices, length ``n``, such that
        ``scores[result]`` is non-increasing and equal scores appear in
        ascending index order.
    """
    scores = np.asarray(scores, dtype=np.float64)
    n = scores.shape[0]
    # lexsort sorts by the last key first: primary key -scores (descending
    # score), secondary key the index (ascending) to break ties.
    return np.lexsort((np.arange(n), -scores))


def rank_positions(scores: np.ndarray) -> np.ndarray:
    """Return the 1-based rank of every node under :func:`ranking`.

    The inverse of the permutation ``ranking`` returns: where ``ranking`` lists
    node indices best-first, this lists, for each node index, the position it
    holds (1 = highest score). The two satisfy
    ``rank_positions(s)[ranking(s)[i]] == i + 1``.

    Args:
        scores: Score per node, shape ``(n,)``.

    Returns:
        An int64 array of shape ``(n,)``; entry ``i`` is node ``i``'s 1-based
        rank, ties broken by ascending index (the same rule as ``ranking``).
    """
    order = ranking(scores)
    n = order.shape[0]
    positions = np.empty(n, dtype=np.int64)
    positions[order] = np.arange(1, n + 1, dtype=np.int64)
    return positions


def top_k_overlap(a: np.ndarray, b: np.ndarray, k: int) -> float:
    """Fraction of the top ``k`` shared by two score vectors.

    ``|top_k(a) ∩ top_k(b)| / k``, with each top-k set taken from
    :func:`ranking` (so ties break by ascending index). A value of 1.0 means
    the two measures pick the same set of ``k`` nodes, regardless of their
    order within it.

    Args:
        a: First score vector, shape ``(n,)``.
        b: Second score vector, same shape as ``a``.
        k: Size of the top set to compare; must be in ``1 .. n``.

    Returns:
        The overlap as a float in ``[0, 1]``.

    Raises:
        ValueError: If the vectors differ in length, or ``k`` is not in
            ``1 .. n``.
    """
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    if a.shape != b.shape:
        raise ValueError(f"a and b must have the same shape, got {a.shape} and {b.shape}")
    n = a.shape[0]
    if not (1 <= k <= n):
        raise ValueError(f"k must be in 1..{n}, got {k}")

    top_a = set(int(i) for i in ranking(a)[:k])
    top_b = set(int(i) for i in ranking(b)[:k])
    return len(top_a & top_b) / k


def kendall_tau(
    a: np.ndarray,
    b: np.ndarray,
    subset: np.ndarray | None = None,
) -> float:
    """Kendall's tau-b between two score vectors, optionally over a subset.

    A rank-correlation in ``[-1, 1]``: 1 for identical orderings, -1 for
    reversed. The tau-b variant corrects for ties, which matter here because
    in-degree is a small integer with many ties. This is the one place the
    project leans on :mod:`scipy.stats` rather than its own code: the
    "from scratch" claim is about the PageRank solver, not about a standard
    statistic, so using ``scipy.stats.kendalltau`` is in scope.

    Args:
        a: First score vector, shape ``(n,)``.
        b: Second score vector, same shape as ``a``.
        subset: Optional index or boolean array selecting the nodes to compare;
            ``None`` compares all nodes.

    Returns:
        The tau-b statistic as a float.

    Raises:
        ValueError: If the vectors differ in length.
    """
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    if a.shape != b.shape:
        raise ValueError(f"a and b must have the same shape, got {a.shape} and {b.shape}")
    if subset is not None:
        a = a[subset]
        b = b[subset]
    return float(scipy.stats.kendalltau(a, b).statistic)

**Structure.** Strongly connected components tell which part of the graph is closed, meaning no edge leaves it, and which ASes are transient. This is used to describe the real graph and does not enter the PageRank computation.

In [ ]:
@dataclass(frozen=True)
class GraphStructure:
    """The strongly/weakly connected decomposition of a directed graph.

    Attributes:
        n_weak_components: Number of weakly connected components (ignoring edge
            direction).
        n_strong_components: Number of strongly connected components (SCCs).
        labels: SCC label per node, int, shape ``(n,)``; nodes with the same
            label are in the same SCC.
        closed: Boolean per SCC, shape ``(n_strong_components,)``. ``True`` when
            no edge leaves that SCC (a closed / recurrent class).
    """

    n_weak_components: int
    n_strong_components: int
    labels: np.ndarray
    closed: np.ndarray

    @property
    def n_closed_classes(self) -> int:
        """How many SCCs are closed (recurrent classes)."""
        return int(self.closed.sum())

    @property
    def transient_mask(self) -> np.ndarray:
        """Per-node boolean: ``True`` when the node's SCC is not closed.

        A transient node is one a random surfer can leave and never return to.
        """
        return ~self.closed[self.labels]

    @property
    def largest_closed_class_size(self) -> int:
        """Number of nodes in the largest closed SCC, 0 if there is none."""
        if self.n_closed_classes == 0:
            return 0
        counts = np.bincount(self.labels, minlength=self.n_strong_components)
        return int(counts[self.closed].max())


def graph_structure(graph: DirectedGraph) -> GraphStructure:
    """Compute the SCC decomposition and closed classes of ``graph``.

    The graph stores ``P^T`` (entry ``[head, tail]`` for an edge
    ``tail -> head``); its transpose recovers the adjacency pattern
    ``[tail, head]``, which is what ``connected_components`` needs. An SCC is
    *closed* when no edge crosses from it to a node in a different SCC: taking
    the edge endpoints ``(tail, head)``, an SCC is open as soon as one of its
    edges lands in another SCC.

    Args:
        graph: The directed graph to analyse.

    Returns:
        A :class:`GraphStructure` with weak- and strong-component counts, the
        per-node SCC labels, and the closed/open flag per SCC.

    Raises:
        ValueError: If ``graph.n == 0``.
    """
    n = graph.n
    if n == 0:
        raise ValueError("cannot analyse the structure of an empty graph (n == 0)")

    # transpose is P^T with a non-zero at [head, tail]; transposing it back
    # gives the adjacency pattern (non-zero at [tail, head]).
    adjacency = graph.transpose.T.tocoo()

    n_strong, labels = connected_components(
        adjacency, directed=True, connection="strong"
    )
    n_weak, _ = connected_components(
        adjacency, directed=True, connection="weak"
    )

    # An SCC is closed unless one of its edges crosses into another SCC.
    closed = np.ones(n_strong, dtype=bool)
    tail_label = labels[adjacency.row]
    head_label = labels[adjacency.col]
    crossing = tail_label != head_label
    closed[tail_label[crossing]] = False

    return GraphStructure(
        n_weak_components=int(n_weak),
        n_strong_components=int(n_strong),
        labels=labels,
        closed=closed,
    )

**The course construction in dense form.** These five functions build $A$, $H$, $S$ and $G$ as dense matrices and take $\pi$ as the dominant left eigenvector of $G$. They are for small graphs only. They are used in section 4 to check the sparse solver against the definition.

In [ ]:
def adjacency_matrix(
    edges: Iterable[tuple[int, int]] | np.ndarray, n: int
) -> np.ndarray:
    """Dense adjacency matrix ``A`` in the course's row convention.

    ``A[i, j] = 1`` when there is an edge ``i -> j`` and 0 otherwise, so the
    out-links of node ``i`` are the non-zeros of row ``i``. This is the matrix
    the lecture calls ``A``; duplicate edges collapse to a single 1 and
    self-loops are kept as written (the lecture examples contain neither).

    Args:
        edges: Iterable of ``(tail, head)`` pairs or an ``(m, 2)`` int array,
            each read as an edge ``tail -> head``.
        n: Number of nodes; indices must lie in ``[0, n)``.

    Returns:
        A float64 array of shape ``(n, n)`` with entries 0 or 1.

    Raises:
        ValueError: If ``n < 0`` or an endpoint is outside ``[0, n)``.
    """
    if n < 0:
        raise ValueError(f"n must be non-negative, got {n}")
    arr = np.asarray(list(edges), dtype=np.int64).reshape(-1, 2) if not isinstance(
        edges, np.ndarray
    ) else np.asarray(edges, dtype=np.int64).reshape(-1, 2)

    a = np.zeros((n, n), dtype=np.float64)
    if arr.size:
        if (arr < 0).any() or (arr >= n).any():
            raise ValueError(f"an edge endpoint is outside [0, {n})")
        a[arr[:, 0], arr[:, 1]] = 1.0
    return a


def hyperlink_matrix(a: np.ndarray) -> np.ndarray:
    """Row-stochastic hyperlink matrix ``H = diag(D)^-1 A``.

    Each row of ``A`` is divided by that node's out-degree, so a non-dangling
    row of ``H`` sums to 1 and distributes a surfer's next step uniformly over
    the out-links. A dangling node (an all-zero row of ``A``) has out-degree 0;
    dividing by ``max(out_degree, 1)`` leaves its row all zero, which is why
    ``H`` is only *sub*-stochastic in general and needs the repair in
    :func:`stochastic_matrix`. This is the lecture's ``H``.

    Args:
        a: An ``(n, n)`` adjacency matrix, as from :func:`adjacency_matrix`.

    Returns:
        A float64 ``(n, n)`` array; row ``i`` sums to 1 unless node ``i`` is
        dangling, in which case it is all zero.
    """
    out_degree = a.sum(axis=1)
    # Divide by max(out_degree, 1): dangling rows (out_degree 0) stay zero
    # rather than becoming NaN, exactly as the sub-stochastic H requires.
    denom = np.where(out_degree > 0, out_degree, 1.0)
    return a / denom[:, None]


def stochastic_matrix(h: np.ndarray) -> np.ndarray:
    """Stochastic matrix ``S = H + (1/N) w 1^T`` repairing dangling rows.

    The lecture makes ``H`` stochastic by replacing each dangling (all-zero)
    row with the uniform distribution ``1/N``: ``w`` is the 0/1 indicator of
    dangling rows and ``w 1^T`` the outer product that writes ``1/N`` into each
    such row. Every row of ``S`` then sums to 1. This is the lecture's ``S``.

    Args:
        h: An ``(n, n)`` hyperlink matrix from :func:`hyperlink_matrix`.

    Returns:
        A row-stochastic float64 ``(n, n)`` array.
    """
    n = h.shape[0]
    row_sums = h.sum(axis=1)
    w = (row_sums == 0).astype(np.float64)  # 1 for dangling rows
    return h + np.outer(w, np.full(n, 1.0 / n))


def google_matrix(s: np.ndarray, alpha: float) -> np.ndarray:
    """Google matrix ``G = alpha S + (1 - alpha)/N 1 1^T``.

    The teleportation step mixes the stochastic matrix ``S`` with the uniform
    jump ``(1/N) 1 1^T`` in proportion ``alpha`` to ``1 - alpha``. With
    ``0 < alpha < 1`` every entry of ``G`` is strictly positive, so ``G`` is
    primitive (irreducible and aperiodic); by Perron-Frobenius it then has a
    unique positive dominant left eigenvector with eigenvalue 1, which is
    PageRank. This is the lecture's ``G``.

    Args:
        s: An ``(n, n)`` stochastic matrix from :func:`stochastic_matrix`.
        alpha: Damping factor, ``0 <= alpha <= 1``.

    Returns:
        A row-stochastic float64 ``(n, n)`` array; strictly positive when
        ``0 < alpha < 1``.
    """
    n = s.shape[0]
    return alpha * s + (1.0 - alpha) / n * np.ones((n, n), dtype=np.float64)


def pagerank_eigenvector(g: np.ndarray) -> np.ndarray:
    """PageRank as the dominant left eigenvector of ``G``, normalised.

    The lecture defines PageRank as the row vector ``pi`` with ``pi G = pi``
    and ``pi 1 = 1``: the left eigenvector of ``G`` for eigenvalue 1. A left
    eigenvector of ``G`` is a right eigenvector of ``G^T``, so this takes the
    eigendecomposition of ``G^T``, selects the eigenvalue closest to 1, keeps
    the real part, fixes the sign to make the vector non-negative, and
    normalises it to sum 1. For a primitive ``G`` that eigenvalue is exactly 1
    and the vector is strictly positive (Perron-Frobenius).

    Args:
        g: An ``(n, n)`` Google matrix from :func:`google_matrix`.

    Returns:
        A float64 ``(n,)`` probability vector summing to 1.

    Raises:
        ValueError: If the dominant eigenvalue is not 1 within 1e-12, which
            means ``g`` is not a valid (primitive stochastic) Google matrix.
    """
    eigenvalues, eigenvectors = np.linalg.eig(g.T)
    # The dominant eigenvalue of a stochastic matrix is 1; pick the one nearest.
    dominant = int(np.argmin(np.abs(eigenvalues - 1.0)))
    lead = eigenvalues[dominant]
    if abs(lead.real - 1.0) > 1e-12 or abs(lead.imag) > 1e-12:
        raise ValueError(
            f"dominant eigenvalue is {lead!r}, not 1; G is not a valid "
            "primitive stochastic matrix"
        )
    vec = eigenvectors[:, dominant].real
    # Eigenvectors come back with arbitrary sign/scale; make it a distribution.
    total = vec.sum()
    if total == 0.0:
        raise ValueError("dominant eigenvector sums to zero; cannot normalise")
    return vec / total

## 4. Validation

A solver that returns a plausible vector can still be wrong. Four checks, each against an answer that does not come from the solver.

### 4.1 Toy graph against exact fractions

Four nodes, five edges, $\alpha = 17/20$, with no dangling node. The exact PageRank was solved in rational arithmetic, so each entry is a fraction. `ALPHA` is kept as a `Fraction` in this section so that the exact answer and the solver use the same damping.

In [ ]:
ALPHA = Fraction(17, 20)
TOY_EDGES = [(0, 1), (0, 2), (1, 2), (2, 0), (3, 2)]
TOY_N = 4
TOY_PAGERANK_EXACT = (
    Fraction(659, 1769),
    Fraction(27713, 141520),
    Fraction(2789, 7076),
    Fraction(3, 80),
)

In [ ]:
toy = from_edge_list(TOY_EDGES, TOY_N)
toy_result = pagerank_power(toy, alpha=float(ALPHA), tol=1e-14, max_iter=5000)
toy_exact = np.array([float(p) for p in TOY_PAGERANK_EXACT])
print(f"iterations                  {toy_result.iterations}")
print(f"exact fractions sum to 1:   {sum(TOY_PAGERANK_EXACT) == 1}")
print(f"max |solver - exact|        {np.abs(toy_result.vector - toy_exact).max():.2e}")
assert np.abs(toy_result.vector - toy_exact).max() < 1e-12

### 4.2 A graph with a dangling node

The second fixture has four nodes and edges $0\to1$, $0\to2$, $1\to2$, $1\to3$, $2\to3$. Node 3 has no out-link, so it is dangling. Node 0 has no in-link, so it receives only the share that every node gets: teleportation plus its part of node 3's redistributed mass. Its PageRank is derived by hand from the repaired matrix $S$ at $\alpha=17/20$ with uniform teleportation.

**Derivation by hand.** In the repaired matrix $S$, node 3 sends a quarter of its rank to each of the four nodes. Every node therefore gets the same base share $k$: its teleportation share plus its quarter of node 3's rank.

$$k = \frac{1-\alpha}{4} + \alpha\,\frac{p_3}{4} = \frac{3}{80} + \frac{17}{80}\,p_3 .$$

On top of $k$, a node receives the rank of each in-neighbour divided by that neighbour's out-degree:

$$p_0 = k, \qquad p_1 = k + \alpha\,\frac{p_0}{2}, \qquad p_2 = k + \alpha\left(\frac{p_0}{2} + \frac{p_1}{2}\right), \qquad p_3 = k + \alpha\left(\frac{p_1}{2} + p_2\right).$$

I did not solve for $k$ first. Going down the chain, I wrote each $p_i$ as a multiple of $k$:

$$p_0 = k, \qquad p_1 = \tfrac{57}{40}\,k, \qquad p_2 = \tfrac{3249}{1600}\,k, \qquad p_3 = \tfrac{106613}{32000}\,k .$$

The four add up to 1, so $k = 32000/249193$ and

$$p = \frac{(32000,\ 45600,\ 64980,\ 106613)}{249193} \approx (0.128,\ 0.183,\ 0.261,\ 0.428).$$

The equation for $k$ played no part in finding it, which makes it a free check: putting $p_3$ back in gives exactly $32000/249193$. Node 0 has no in-link and gets only the base share, so it ranks last. Node 3 ranks first because it collects all of node 2's rank and half of node 1's.

The cell below runs the solver on the fixture and compares it with the hand-derived exact vector. It also checks the claim of section 2 numerically: repairing the dangling row, and leaving the matrix substochastic and normalising the solution at the end, give the same vector.

In [ ]:
DANGLING_EDGES = [(0, 1), (0, 2), (1, 2), (1, 3), (2, 3)]
DANGLING_N = 4
DANGLING_PAGERANK_EXACT = (
    Fraction(32000, 249193),
    Fraction(45600, 249193),
    Fraction(64980, 249193),
    Fraction(106613, 249193),
)

In [ ]:
dangling_graph = from_edge_list(DANGLING_EDGES, DANGLING_N)
print(f"dangling nodes in the fixture: {np.flatnonzero(dangling_graph.dangling).tolist()}")
dangling_result = pagerank_power(dangling_graph, alpha=float(ALPHA), tol=1e-14, max_iter=5000)
v_sparse = dangling_result.vector

# Convention 1: repair the matrix (S), build G, take the eigenvector.
h = hyperlink_matrix(adjacency_matrix(DANGLING_EDGES, DANGLING_N))
v_repaired = pagerank_eigenvector(google_matrix(stochastic_matrix(h), float(ALPHA)))

# Convention 2: keep H substochastic, solve z = (1-a) v + a H^T z, normalise.
n_fix = DANGLING_N
z = np.linalg.solve(
    np.eye(n_fix) - float(ALPHA) * h.T, (1 - float(ALPHA)) * np.full(n_fix, 1 / n_fix)
)
v_normalised = z / z.sum()

print(f"vector from the solver           {np.round(v_sparse, 6)}")
print(f"sum                              {v_sparse.sum():.12f}")
print(f"mass lost before normalising     {1 - z.sum():.4f}")
print(f"max |solver - repaired matrix|   {np.abs(v_sparse - v_repaired).max():.2e}")
print(f"max |solver - normalised solve|  {np.abs(v_sparse - v_normalised).max():.2e}")
assert np.abs(v_sparse - v_repaired).max() < 1e-12
assert np.abs(v_sparse - v_normalised).max() < 1e-12

if DANGLING_PAGERANK_EXACT is None:
    print("hand-derived exact vector: not filled in yet")
else:
    exact = np.array([float(p) for p in DANGLING_PAGERANK_EXACT])
    print(f"max |solver - hand-derived|      {np.abs(v_sparse - exact).max():.2e}")
    assert np.abs(v_sparse - exact).max() < 1e-12

### 4.3 Examples from the lecture

The lecture gives a five-node graph at $\alpha=0.85$ with the printed result $(0.1219, 0.1219, 0.1737, 0.2475, 0.3350)$. Node V (index 0) is dangling. The table shows the printed values, the sparse solver, and the dense eigenvector of $G$.

In [ ]:
FIVE_EDGES = [(1, 2), (1, 3), (2, 3), (2, 4), (3, 4), (4, 0), (4, 1), (4, 2), (4, 3)]
FIVE_N = 5
FIVE_PRINTED = np.array([0.1219, 0.1219, 0.1737, 0.2475, 0.3350])

five_sparse = pagerank_power(
    from_edge_list(FIVE_EDGES, FIVE_N), alpha=0.85, tol=1e-14, max_iter=5000
).vector
five_dense = pagerank_eigenvector(
    google_matrix(
        stochastic_matrix(hyperlink_matrix(adjacency_matrix(FIVE_EDGES, FIVE_N))), 0.85
    )
)
five_table = pd.DataFrame(
    {"printed": FIVE_PRINTED, "sparse solver": five_sparse, "dense eigenvector": five_dense},
    index=list("VWXYZ"),
)
print(five_table.round(4))
print(f"max |sparse - dense|   {np.abs(five_sparse - five_dense).max():.2e}")
print(f"max |sparse - printed| {np.abs(five_sparse - FIVE_PRINTED).max():.2e} (the slide prints 4 decimals)")
assert np.abs(five_sparse - five_dense).max() < 1e-12
assert np.abs(five_sparse - FIVE_PRINTED).max() < 5e-5

The lecture's second example has no teleportation and four nodes W, X, Y, Z. The graph is strongly connected, so with $\alpha=1$ the matrix $S$ has a unique stationary vector, here exactly $(4,6,9,12)/31$. The sparse solver rejects $\alpha=1$, so the dense construction is used. The example also shows why in-degree is not PageRank: Y has the largest in-degree, Z has the largest PageRank.

In [ ]:
FOUR_EDGES = [(0, 1), (0, 2), (1, 2), (1, 3), (2, 3), (3, 0), (3, 1), (3, 2)]
FOUR_EXACT = (Fraction(4, 31), Fraction(6, 31), Fraction(9, 31), Fraction(12, 31))

four_dense = pagerank_eigenvector(
    google_matrix(stochastic_matrix(hyperlink_matrix(adjacency_matrix(FOUR_EDGES, 4))), 1.0)
)
four_in_degree = from_edge_list(FOUR_EDGES, 4).in_degree
print(f"max |dense - exact|   {np.abs(four_dense - np.array([float(p) for p in FOUR_EXACT])).max():.2e}")
print(f"in-degree             {dict(zip('WXYZ', four_in_degree.tolist()))}  -> leader {'WXYZ'[int(four_in_degree.argmax())]}")
print(f"PageRank              {dict(zip('WXYZ', np.round(four_dense, 4).tolist()))}  -> leader {'WXYZ'[int(four_dense.argmax())]}")
assert np.abs(four_dense - np.array([float(p) for p in FOUR_EXACT])).max() < 1e-12

### What the validation established

The sparse power method matches exact fractions on the toy graph, matches the two dangling conventions to rounding error on the fixture, and reproduces the lecture's printed values. The dense construction of the course and the sparse solver agree. The solver can now be applied to the real graph.

## 5. The real graph

The snapshot is read and turned into the primary graph: peering `both`, vote customer to provider, all ASes kept.

In [ ]:
snapshot = read_as_rel(AS_REL)
print(f"relationship lines     {len(snapshot.relationships):,}")
print(f"  customer-provider    {snapshot.n_c2p:,}")
print(f"  peer-to-peer         {snapshot.n_p2p:,}")
print(f"input clique           {len(snapshot.clique)} ASes: {list(snapshot.clique)}")
print(f"IXP ASes in header     {len(snapshot.ixp_asns)}")

build = build_edge_list(snapshot, peering="both", vote="customer_to_provider")
graph = from_edge_list(build.edges, build.n)
peering_share = 2 * build.n_p2p_used / graph.n_edges
print()
print(f"primary graph: {graph.n:,} nodes, {graph.n_edges:,} edges")
print(f"duplicate edges dropped {graph.n_duplicates_dropped}, self-loops dropped {graph.n_self_loops_dropped}")
print(f"dangling nodes          {int(graph.dangling.sum())}")
print(f"share of edges that come from peering links: {peering_share:.1%}")
assert graph.n == 80510 and graph.n_edges == 884790
assert int(graph.dangling.sum()) == 0

The peering decision matters because peering supplies most of the edges. Under `none` the graph keeps only the customer-provider edges, and the ASes that have no provider have nowhere to send their vote:

In [ ]:
build_none = build_edge_list(snapshot, peering="none", vote="customer_to_provider")
graph_none = from_edge_list(build_none.edges, build_none.n)
assert np.array_equal(build_none.asns, build.asns)  # same node indices

clique_nodes = index_of(build.asns, np.array(snapshot.clique))
print(f"peering none: {graph_none.n_edges:,} edges, {int(graph_none.dangling.sum())} dangling nodes")
print(f"clique members among the dangling nodes: {int(graph_none.dangling[clique_nodes].sum())} of {len(clique_nodes)}")
assert int(graph_none.dangling.sum()) == 494

[AUTHOR: modelling]

**Reducibility.** A random surfer without teleportation would eventually be stuck in a closed class, a set of ASes with no edge leaving it. The strongly connected components show how the primary graph is built.

In [ ]:
structure = graph_structure(graph)
transient = structure.transient_mask
print(f"weakly connected components      {structure.n_weak_components}")
print(f"strongly connected components    {structure.n_strong_components:,}")
print(f"closed classes                   {structure.n_closed_classes}")
print(f"size of the largest closed class {structure.largest_closed_class_size:,}")
print(f"transient ASes                   {int(transient.sum()):,} of {graph.n:,}")
assert structure.n_closed_classes == 1 and structure.largest_closed_class_size == 20401

The graph is connected as an undirected graph, but it is not strongly connected: there is one closed class, and the other ASes are transient. Without teleportation, all the mass would end up inside the closed class and every transient AS would get 0. Teleportation is what gives them rank.

**Degree distribution.** In-degree and out-degree, as complementary cumulative distributions on log-log axes. Both are heavy-tailed: the median in-degree is 0 and the median out-degree is 2, while a few ASes have thousands of links.

In [ ]:
def ccdf(values: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Points (x, P(X >= x)) of the empirical survival function."""
    ordered = np.sort(values)
    unique, first = np.unique(ordered, return_index=True)
    return unique, 1.0 - first / ordered.size


fig, ax = plt.subplots(figsize=(6.5, 4.5))
for degree, label in ((graph.in_degree, "in-degree"), (graph.out_degree, "out-degree")):
    x, y = ccdf(degree[degree > 0])
    ax.loglog(x, y, ".", markersize=4, label=label)
ax.set_xlabel("degree d")
ax.set_ylabel("share of ASes (degree >= 1) with degree >= d")
ax.set_title("Degree distribution of the AS graph (peering both)")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()
print(f"max in-degree {int(graph.in_degree.max()):,}, max out-degree {int(graph.out_degree.max()):,}")
print(f"median in-degree {int(np.median(graph.in_degree))}, median out-degree {int(np.median(graph.out_degree))}")

## 6. Results: the ranking

PageRank at $\alpha=0.85$, to a residual below $10^{-10}$.

In [ ]:
ALPHA_MAIN = 0.85
TOL = 1e-10

result = pagerank_power(graph, alpha=ALPHA_MAIN, tol=TOL)
pr = result.vector
print(f"iterations      {result.iterations}")
print(f"converged       {result.converged}")
print(f"final residual  {result.residuals[-1]:.2e}")
print(f"sum of vector   {pr.sum():.15f}")
print(f"min / max       {pr.min():.2e} / {pr.max():.2e}  (uniform would be {1 / graph.n:.2e})")
assert result.converged and abs(pr.sum() - 1) < 1e-12

The top 20, with the facts about each AS taken from the snapshot: how many customers, providers and peers it has, and whether it is in the clique.

In [ ]:
rel = snapshot.relationships
c2p_rows = rel[rel[:, 2] == C2P]
p2p_rows = rel[rel[:, 2] == P2P]


def count_per_node(asn_column: np.ndarray) -> np.ndarray:
    """How many times each node's ASN appears in a column of ASNs."""
    counts = pd.Series(asn_column).value_counts()
    return counts.reindex(build.asns, fill_value=0).to_numpy()


n_customers = count_per_node(c2p_rows[:, 0])  # as1 is the provider
n_providers = count_per_node(c2p_rows[:, 1])  # as2 is the customer
n_peers = count_per_node(np.concatenate([p2p_rows[:, 0], p2p_rows[:, 1]]))
is_clique = np.zeros(graph.n, dtype=bool)
is_clique[clique_nodes] = True

in_degree = graph.in_degree.astype(np.float64)
pr_rank = rank_positions(pr)
in_rank = rank_positions(in_degree)
order_pr = ranking(pr)


def as_row(node: int) -> dict:
    return {
        "rank": int(pr_rank[node]),
        "ASN": int(build.asns[node]),
        "PageRank": float(pr[node]),
        "in-degree": int(graph.in_degree[node]),
        "in-degree rank": int(in_rank[node]),
        "out-degree": int(graph.out_degree[node]),
        "customers": int(n_customers[node]),
        "providers": int(n_providers[node]),
        "peers": int(n_peers[node]),
        "clique": "yes" if is_clique[node] else "no",
    }


top20 = pd.DataFrame([as_row(int(node)) for node in order_pr[:20]])
top20

The PageRank values are spread over more than three orders of magnitude. The smallest value, 1.9e-6, equals $(1-\alpha)/N$, the share an AS gets from teleportation alone, which is what an AS with no in-link receives.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5))
x, y = ccdf(pr)
ax.loglog(x, y, ".", markersize=4)
ax.axvline(1 / graph.n, color="gray", linestyle=":", label="uniform value 1/N")
ax.set_xlabel("PageRank p")
ax.set_ylabel("share of ASes with PageRank >= p")
ax.set_title("Distribution of PageRank (alpha 0.85)")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

Facts about the top of the ranking, as numbers only:

In [ ]:
top20_nodes = order_pr[:20]
hubs = int((n_peers[top20_nodes] > n_customers[top20_nodes]).sum())
print(f"top 20: {int(is_clique[top20_nodes].sum())} are clique members")
print(f"top 20: {hubs} have more peers than customers")
print(f"top 20: median customers {int(np.median(n_customers[top20_nodes]))}, median peers {int(np.median(n_peers[top20_nodes]))}, median providers {int(np.median(n_providers[top20_nodes]))}")
print(f"top 20: hold {pr[top20_nodes].sum():.1%} of the total PageRank")
clique_ranks = np.sort(pr_rank[clique_nodes])
print(f"clique members in the top 100: {int((clique_ranks <= 100).sum())} of {len(clique_nodes)}")
print(f"worst-placed clique member: rank {int(clique_ranks[-1])}")

[AUTHOR: top]

## 7. PageRank versus in-degree

In-degree counts the in-links of an AS. PageRank counts them too, but weights each by the rank of the sender and divides it by the sender's out-degree. The two are close on this graph, so the disagreements are the informative part.

In [ ]:
union_top = np.union1d(order_pr[:1000], ranking(in_degree)[:1000])
comparison = pd.DataFrame(
    {
        "value": {
            "top-10 overlap": top_k_overlap(pr, in_degree, 10),
            "top-50 overlap": top_k_overlap(pr, in_degree, 50),
            "top-100 overlap": top_k_overlap(pr, in_degree, 100),
            "Kendall tau-b, all ASes": kendall_tau(pr, in_degree),
            "Kendall tau-b, top-1000 union": kendall_tau(pr, in_degree, subset=union_top),
        }
    }
)
comparison.round(3)

In [ ]:
mask = in_degree >= 1
fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.scatter(in_degree[mask], pr[mask], s=4, alpha=0.25, label="AS")
ax.scatter(in_degree[is_clique], pr[is_clique], s=28, color="crimson", label="clique member")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("in-degree")
ax.set_ylabel("PageRank")
ax.set_title("PageRank against in-degree (peering both)")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

**Where they disagree.** The comparison is restricted to the 1000 best ASes by either measure, so that it is between networks that matter to at least one of them. For each AS the table gives the shift in rank and what its in-neighbours look like. The last column is the average amount that one in-link delivers, the term $\pi_j / d_j^{\text{out}}$ of the update averaged over the in-neighbours $j$.

In [ ]:
transpose = graph.transpose.tocsr()  # row `node` lists the in-neighbours of `node`


def in_link_contribution(node: int) -> float:
    """Average of pr[j] / out_degree[j] over the in-neighbours j of `node`."""
    tails = transpose.indices[transpose.indptr[node] : transpose.indptr[node + 1]]
    return float((pr[tails] / graph.out_degree[tails]).mean())


shift = pr_rank - in_rank  # negative: PageRank ranks the AS better than in-degree does


def disagreements(candidates: np.ndarray, direction: str, k: int = 10) -> pd.DataFrame:
    chosen = sorted(candidates.tolist(), key=lambda node: (-abs(int(shift[node])), node))[:k]
    rows = []
    for node in chosen:
        row = as_row(node)
        row["rank shift"] = abs(int(shift[node]))
        row["per in-link"] = in_link_contribution(node)
        row["better by"] = direction
        rows.append(row)
    return pd.DataFrame(rows)


higher_pr = disagreements(union_top[shift[union_top] < 0], "PageRank")
higher_in = disagreements(union_top[shift[union_top] > 0], "in-degree")
pd.concat([higher_pr, higher_in], ignore_index=True)[
    ["better by", "ASN", "rank", "in-degree rank", "rank shift", "in-degree", "peers", "customers", "per in-link"]
]

**Vote dilution.** An in-link from a large AS with many out-links delivers little, because the sender splits its rank over all its out-links. An AS can have a high in-degree and still receive little rank if its in-links come from senders that are small or that link to many others.

In [ ]:
print(f"median rank per in-link, ASes that PageRank ranks higher: {higher_pr['per in-link'].median():.2e}")
print(f"median rank per in-link, ASes that in-degree ranks higher: {higher_in['per in-link'].median():.2e}")

### What the comparison established

On this graph PageRank and in-degree give almost the same top of the ranking, and they differ on a small set of ASes. The ASes that PageRank puts higher get more rank from each in-link than the ASes that in-degree puts higher.

## 8. Dangling nodes on the real graph

Under the primary graph no AS is dangling, so the dangling term of the solver is inactive there. It becomes active when peering links are left out. The same solver is run on that graph without change.

In [ ]:
result_none = pagerank_power(graph_none, alpha=ALPHA_MAIN, tol=TOL)
pr_none = result_none.vector
dangling_mass = float(pr_none[graph_none.dangling].sum())
print(f"iterations                      {result_none.iterations}")
print(f"final residual                  {result_none.residuals[-1]:.2e}")
print(f"sum of vector                   {pr_none.sum():.15f}")
print(f"dangling ASes                   {int(graph_none.dangling.sum())}")
print(f"PageRank mass on dangling ASes  {dangling_mass:.4f}")
print(f"clique members that dangle      {int(graph_none.dangling[clique_nodes].sum())} of {len(clique_nodes)}")
print()
for k in (10, 20, 50, 100):
    print(f"top-{k:<3d} overlap, both vs none: {top_k_overlap(pr, pr_none, k):.2f}")
print(f"Kendall tau-b, both vs none: {kendall_tau(pr, pr_none):.3f}")
assert result_none.converged and abs(pr_none.sum() - 1) < 1e-12

Without peering, the clique is among the ASes with no out-link, and the top of the ranking changes a lot: the top-20 overlap between the two conventions is only 0.25. This is where the dangling treatment matters on real data. The mass of the dangling ASes is not lost: it is spread uniformly over all ASes at every step, and the vector still sums to 1.

In [ ]:
rank_none = rank_positions(pr_none)
pd.DataFrame(
    {
        "ASN": build.asns[clique_nodes],
        "rank, peering both": pr_rank[clique_nodes],
        "rank, peering none": rank_none[clique_nodes],
    }
).sort_values("rank, peering none").reset_index(drop=True)

In [ ]:
node_2828 = int(index_of(build.asns, np.array([2828]))[0])
print(
    f"AS 2828: {n_customers[node_2828]} customers, {n_providers[node_2828]} providers, "
    f"{n_peers[node_2828]} peers; rank {pr_rank[node_2828]} with peering both, "
    f"{rank_none[node_2828]} with peering none"
)

Under `none` an AS receives rank along links only from its customers, and AS 2828 has 2.

## 9. Extensions

Two short extensions, after the core answers above.

### 9.1 Customer cone as an external reference

CAIDA also ships, for the same snapshot, the *customer cone* of every AS: the ASes reachable from it by going down customer links. It is an independent size measure, computed from paths and not from this graph's edges, so it can serve as a reference ranking. The code below reads the cone file and, as a check, recomputes cones directly from the relationships.

In [ ]:
@dataclass(frozen=True)
class CustomerCones:
    """Per-AS customer-cone sizes read from a CAIDA ``ppdc-ases`` file.

    The ``ppdc-ases`` file ships with the same snapshot as the ``as-rel`` file
    and lists, one line per AS, the AS followed by every AS in its
    provider-peer-customer cone -- the set reachable by following
    provider-to-customer links downwards (see CAIDA's description of
    ``serial-1`` and Luckie et al., IMC 2013). It is CAIDA's own "how big is
    this AS" ordering, independent of anything this project computes, so it
    serves as an external reference for the ranking.

    Only the cone *size* per AS is kept: the member lists run to tens of
    thousands of ASNs for the clique and are not needed downstream, so they are
    counted per line and discarded during the parse.

    ``cone_size`` counts the distinct ASNs on a line including the leading AS
    itself. On the shipped snapshots every line lists its own AS among the
    members (``self_listed_share == 1.0``), but defining the size over the
    distinct set makes it independent of that convention: an AS with no
    customers has cone size 1 (itself) either way.

    Attributes:
        path: The path the cones were read from.
        clique: ASNs on the ``# inferred clique:`` header line, in file order.
            Empty when the header leaves it blank (as on this snapshot).
        total_size: The value on the ``# total size:`` header line, i.e. the
            number of ASes the file claims to cover.
        asns: ``(m,)`` int64, the ASNs with a cone, sorted ascending. Index
            ``i`` is ASN ``asns[i]``.
        cone_size: ``(m,)`` int64 aligned with ``asns``; distinct ASNs on each
            AS's line including the AS itself.
        n_comment_lines: Count of lines beginning with ``#``.
        self_listed_share: Share of data lines whose leading ASN also appears
            among its listed members.
    """

    path: str
    clique: tuple[int, ...]
    total_size: int
    asns: np.ndarray
    cone_size: np.ndarray
    n_comment_lines: int
    self_listed_share: float


def read_ppdc_ases(path: str | os.PathLike) -> CustomerCones:
    """Parse a CAIDA ``ppdc-ases`` file into per-AS cone sizes.

    Reads the provider-peer-customer cone file (bz2 when the suffix is
    ``.bz2``, plain text otherwise). The header carries two comment lines,
    ``# inferred clique:`` (optionally followed by the clique ASNs; blank on
    this snapshot) and ``# total size:``. Each remaining line is an AS followed
    by the ASNs in its cone, space separated.

    The cone size stored is the number of *distinct* ASNs on the line including
    the leading AS, so that an AS with no customers has size 1 regardless of
    whether the file repeats the leading AS in the member list. The share of
    lines that do repeat it is reported in ``self_listed_share`` so the
    convention on a given file is measurable rather than assumed.

    Member lists are counted and discarded per line; they are never all held in
    memory at once. ``str.split`` on each line is adequate (a few seconds on the
    80k-line snapshot even though the clique lines carry tens of thousands of
    members).

    Args:
        path: Path to the ``ppdc-ases`` file.

    Returns:
        The parsed :class:`CustomerCones`. ``asns`` is sorted ascending and
        ``cone_size`` is aligned with it.

    Raises:
        ValueError: On a line with a non-integer token. The message names the
            1-based line number.
    """
    clique: tuple[int, ...] = ()
    total_size = 0
    n_comment_lines = 0
    n_self_listed = 0

    asn_list: list[int] = []
    size_list: list[int] = []

    with _open_text(path) as handle:
        for k, raw in enumerate(handle, start=1):
            line = raw.rstrip("\n")

            if line.startswith("#"):
                n_comment_lines += 1
                if line.startswith("# inferred clique:"):
                    body = line[len("# inferred clique:"):].strip()
                    if body:
                        clique = tuple(int(t) for t in body.split())
                elif line.startswith("# total size:"):
                    total_size = int(line[len("# total size:"):].strip())
                continue

            if not line.strip():
                continue

            try:
                tokens = [int(t) for t in line.split()]
            except ValueError:
                raise ValueError(
                    f"line {k}: malformed cone line, non-integer token in "
                    f"{line!r}"
                ) from None

            lead = tokens[0]
            members = tokens[1:]
            # Distinct ASNs on the line, including the leading AS. Using a set
            # absorbs both the self-listing and any duplicate member, so the
            # size is "distinct ASNs in the cone" under either convention.
            distinct = set(tokens)
            asn_list.append(lead)
            size_list.append(len(distinct))
            if lead in members:
                n_self_listed += 1

    asns = np.array(asn_list, dtype=np.int64)
    cone_size = np.array(size_list, dtype=np.int64)
    # Sort by ASN so the array is a sorted index like EdgeListBuild.asns.
    order = np.argsort(asns, kind="stable")
    asns = asns[order]
    cone_size = cone_size[order]

    n_lines = len(asn_list)
    self_listed_share = (n_self_listed / n_lines) if n_lines else 0.0

    return CustomerCones(
        path=os.fspath(path),
        clique=clique,
        total_size=total_size,
        asns=asns,
        cone_size=cone_size,
        n_comment_lines=n_comment_lines,
        self_listed_share=self_listed_share,
    )


def customer_cone_sizes(
    snapshot: AsRelSnapshot, asns: Iterable[int]
) -> np.ndarray:
    """Compute customer-cone sizes directly from the relationship file.

    The customer cone of an AS is the set of ASes reachable from it by
    following provider-to-customer links downwards (the ``-1`` rows, where
    ``as1`` is the provider and ``as2`` its customer). Its size is the number
    of *distinct* ASes in that set including the AS itself, so an AS with no
    customers has cone size 1.

    This computes the *recursive* cone over the inferred provider-to-customer
    links: every AS reachable downwards, regardless of whether any observed
    BGP path passes through the root. CAIDA's shipped ``ppdc-ases`` file is a
    different object, the *provider/peer-observed* cone (Luckie et al., IMC
    2013): an AS counts in the root's cone only if a path seen from one of the
    root's providers or peers reaches it through the root via customer links.
    Every observed-cone member is reachable by customer links, so the recursive
    cone computed here is an **upper bound** on (a superset of) the shipped
    cone, never smaller. The two therefore differ whenever an inferred customer
    link was never seen carrying the root's routes; a single such link into a
    large network inflates the recursive cone by that network's whole cone,
    which is why CAIDA observes cones from paths rather than recursing over
    inferred links.

    A provider-to-customer adjacency is built once from the ``-1`` rows; each
    requested ASN is then expanded by breadth-first search. The visited set
    makes the traversal count every reached AS once and terminates on the
    cycles that AS relationships can contain.

    Args:
        snapshot: The parsed relationship snapshot.
        asns: The ASNs to compute cone sizes for, in the order the result
            should follow.

    Returns:
        An int64 array aligned with ``asns``; entry ``i`` is the customer-cone
        size of ``asns[i]`` (distinct ASes reachable downwards, including
        itself, minimum 1).
    """
    rel = snapshot.relationships
    c2p = rel[rel[:, 2] == C2P]
    providers = c2p[:, 0]
    customers = c2p[:, 1]

    # provider -> list of direct customers, built once. Group by provider via a
    # sort so the adjacency is a slice per provider rather than a Python dict of
    # lists rebuilt per edge.
    order = np.argsort(providers, kind="stable")
    sorted_prov = providers[order]
    sorted_cust = customers[order]
    unique_prov, starts = np.unique(sorted_prov, return_index=True)
    ends = np.append(starts[1:], sorted_prov.shape[0])
    adjacency: dict[int, np.ndarray] = {
        int(p): sorted_cust[s:e]
        for p, s, e in zip(unique_prov, starts, ends)
    }

    from collections import deque

    sizes: list[int] = []
    for asn in asns:
        root = int(asn)
        visited = {root}
        queue = deque([root])
        while queue:
            node = queue.popleft()
            children = adjacency.get(node)
            if children is None:
                continue
            for child in children:
                c = int(child)
                if c not in visited:
                    visited.add(c)
                    queue.append(c)
        sizes.append(len(visited))

    return np.array(sizes, dtype=np.int64)

In [ ]:
cones = read_ppdc_ases(PPDC)
cone_size = np.ones(graph.n)  # an AS with no cone line has no customers: size 1
position = np.clip(np.searchsorted(cones.asns, build.asns), 0, cones.asns.shape[0] - 1)
listed = cones.asns[position] == build.asns
cone_size[listed] = cones.cone_size[position[listed]]
print(f"ASes with a cone line: {int(listed.sum()):,} of {graph.n:,}")

cone_top1000 = ranking(cone_size)[:1000]


def versus_cone(scores: np.ndarray, name: str) -> dict:
    union = np.union1d(cone_top1000, ranking(scores)[:1000])
    return {
        "measure": name,
        "top-10 overlap": top_k_overlap(cone_size, scores, 10),
        "top-100 overlap": top_k_overlap(cone_size, scores, 100),
        "tau-b, top-1000 union": kendall_tau(cone_size, scores, subset=union),
    }


pd.DataFrame(
    [
        versus_cone(pr, "PageRank, peering both"),
        versus_cone(pr_none, "PageRank, peering none"),
        versus_cone(in_degree, "in-degree"),
    ]
).round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
for ax, scores, title in (
    (axes[0], pr, "peering both"),
    (axes[1], pr_none, "peering none"),
):
    ax.scatter(cone_size, scores, s=4, alpha=0.25)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("customer cone size")
    ax.set_title(f"PageRank, {title}")
    ax.grid(True, which="both", alpha=0.3)
axes[0].set_ylabel("PageRank")
fig.tight_layout()
plt.show()

The cone measures how many networks sit below an AS in the hierarchy. PageRank with peering removed agrees with it most. There the rank flows only up customer-provider links, which is the relation the cone counts. With peering included the agreement drops, which fits peering edges giving rank to ASes whose cone is small. In-degree agrees least.

In [ ]:
union_cone = np.union1d(cone_top1000, order_pr[:1000])
own_cone = customer_cone_sizes(snapshot, build.asns[union_cone].tolist())
print(f"recursive cone >= CAIDA cone for all {union_cone.size:,} ASes checked: {bool((own_cone >= cone_size[union_cone]).all())}")

The two cone definitions are not the same object. CAIDA's cone is the set seen through observed paths, and the recursive one follows every inferred customer link, so it can only be larger. The comparison above uses CAIDA's.

### 9.2 The damping factor

The damping factor $\alpha$ sets how far a surfer follows links before teleporting. For $\alpha = 0.5,\ 0.85,\ 0.99$ the table gives the iterations needed, the estimate $\lceil\ln(10^{-10})/\ln\alpha\rceil$, the observed rate (median ratio of successive residuals over the second half of the run) and how much the ranking moves relative to $\alpha=0.85$.

In [ ]:
rows = []
for alpha in (0.50, 0.85, 0.99):
    run = pagerank_power(graph, alpha=alpha, tol=TOL, max_iter=5000)
    residuals = np.array(run.residuals)
    tail = residuals[residuals.size // 2 :]
    union_alpha = np.union1d(ranking(run.vector)[:1000], order_pr[:1000])
    rows.append(
        {
            "alpha": alpha,
            "iterations": run.iterations,
            "alpha^k estimate": int(np.ceil(np.log(TOL) / np.log(alpha))),
            "observed rate": float(np.median(tail[1:] / tail[:-1])),
            "top-10 overlap with 0.85": top_k_overlap(run.vector, pr, 10),
            "tau-b with 0.85 (top-1000 union)": kendall_tau(run.vector, pr, subset=union_alpha),
        }
    )
    assert rows[-1]["observed rate"] < alpha
pd.DataFrame(rows).round(3)

The estimate holds and is loose: the observed rate is below $\alpha$ because the second eigenvalue of the matrix is smaller than $\alpha$. The top of the ranking changes with $\alpha$: a smaller $\alpha$ puts more weight on teleportation and a larger one on the link structure.

## 10. Limitations and conclusions

**Limitations.**

- The relationships are inferred from BGP paths, not observed. Some of them are wrong.
- The snapshot is one month. The graph changes, and a ranking for another month would differ in its details.
- Peering links are under-observed in BGP data, because they are only announced to the peer and its customers. So the graph under-counts them, and the true share of peering edges is probably above the 81% measured here.
- PageRank here measures position in the graph of agreements. It does not measure traffic, revenue or the reach of an AS in any physical sense.
- A single damping factor, 0.85, is used for the main results, from the web setting. Section 9.2 shows how far the ranking moves for other values.

[AUTHOR: conclusions]

In [ ]:
print(f"total run time of this notebook: {time.perf_counter() - T0:.0f} s")